In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded")

Libraries loaded


In [2]:
df = pd.read_csv('../data/SampleSuperstore.csv', encoding='latin1')

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
df.head()

Shape: (9994, 13)

Columns: ['Ship Mode', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Category', 'Sub-Category', 'Sales', 'Quantity', 'Discount', 'Profit']


,Ship Mode,Segment,Country,City,State,Postal Code,Region,Category,Sub-Category,Sales,Quantity,Discount,Profit
0,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Bookcases,261.9600,2,0.00,41.9136
1,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,731.9400,3,0.00,219.5820
2,Second Class,Corporate,United States,Los Angeles,California,90036,West,Office Supplies,Labels,14.6200,2,0.00,6.8714
3,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Furniture,Tables,957.5775,5,0.45,-383.0310
4,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Office Supplies,Storage,22.3680,2,0.20,2.5164


In [4]:
print("Data Types")
print(df.dtypes)

print("\nMissing Values")
print(df.isnull().sum())

print("\nBasic Stats")
df[['Sales', 'Quantity', 'Discount', 'Profit']].describe()

Data Types
Ship Mode        object
Segment          object
Country          object
City             object
State            object
Postal Code       int64
Region           object
Category         object
Sub-Category     object
Sales           float64
Quantity          int64
Discount        float64
Profit          float64
dtype: object

Missing Values
Ship Mode       0
Segment         0
Country         0
City            0
State           0
Postal Code     0
Region          0
Category        0
Sub-Category    0
Sales           0
Quantity        0
Discount        0
Profit          0
dtype: int64

Basic Stats


,Sales,Quantity,Discount,Profit
count,9994.000000,9994.000000,9994.000000,9994.000000
mean,229.858001,3.789574,0.156203,28.656896
std,623.245101,2.225110,0.206452,234.260108
min,0.444000,1.000000,0.000000,-6599.978000
25%,17.280000,2.000000,0.000000,1.728750
50%,54.490000,3.000000,0.200000,8.666500
75%,209.940000,5.000000,0.200000,29.364000
max,22638.480000,14.000000,0.800000,8399.976000


In [7]:
before = df.shape[0]
df.drop_duplicates(inplace=True)
after = df.shape[0]

print(f"Rows before        : {before}")
print(f"Rows after         : {after}")
print(f"Duplicates removed : {before - after}")

Rows before        : 9994
Rows after         : 9977
Duplicates removed : 17


In [8]:
print("Missing values before cleaning:")
print(df.isnull().sum()[df.isnull().sum() > 0])

df.dropna(subset=['Sales', 'Quantity', 'Profit', 'Discount'], inplace=True)

print("\nAfter cleaning — total missing:", df.isnull().sum().sum())

Missing values before cleaning:
Series([], dtype: int64)

After cleaning — total missing: 0


In [10]:
# Safe Unit Price — avoid division by zero
df['Unit_Price'] = np.where(
    df['Quantity'] != 0,
    df['Sales'] / df['Quantity'],
    np.nan
)

# Safe Profit Margin — avoid division by zero
df['Profit_Margin'] = np.where(
    df['Sales'] != 0,
    (df['Profit'] / df['Sales']) * 100,
    np.nan
)

# Revenue (explicit naming)
df['Revenue'] = df['Sales']

print("Derived columns created")
df[['Sales', 'Quantity', 'Unit_Price', 'Profit', 'Profit_Margin']].head()

Derived columns created


,Sales,Quantity,Unit_Price,Profit,Profit_Margin
0,261.9600,2,130.9800,41.9136,16.00
1,731.9400,3,243.9800,219.5820,30.00
2,14.6200,2,7.3100,6.8714,47.00
3,957.5775,5,191.5155,-383.0310,-40.00
4,22.3680,2,11.1840,2.5164,11.25


In [11]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(subset=['Unit_Price', 'Profit_Margin'], inplace=True)

print("Infinities cleaned")
print("Remaining rows:", df.shape[0])

Infinities cleaned
Remaining rows: 9977


In [12]:
print("Null Check")
print(df[['Unit_Price', 'Profit_Margin']].isnull().sum())

print("\nTop Unit Prices")
print(df['Unit_Price'].sort_values(ascending=False).head())

print("\nTop Profit Margins")
print(df['Profit_Margin'].sort_values(ascending=False).head())

print("\nLowest Profit Margins")
print(df['Profit_Margin'].sort_values().head())

print("\nFull Stats")
print(df[['Unit_Price', 'Profit_Margin']].describe())

Null Check
Unit_Price       0
Profit_Margin    0
dtype: int64

Top Unit Prices
2697    3773.080
6826    3499.990
4190    3499.990
8153    3499.990
2623    2799.992
Name: Unit_Price, dtype: float64

Top Profit Margins
6212    50.0
6250    50.0
1880    50.0
9951    50.0
484     50.0
Name: Profit_Margin, dtype: float64

Lowest Profit Margins
676    -275.0
261    -275.0
9164   -275.0
8766   -275.0
6989   -270.0
Name: Profit_Margin, dtype: float64

Full Stats
        Unit_Price  Profit_Margin
count  9977.000000    9977.000000
mean     60.985982      12.011354
std     143.029806      46.663769
min       0.336000    -275.000000
25%       5.472000       7.500000
50%      16.272000      27.000000
75%      63.940000      36.250000
max    3773.080000      50.000000


In [13]:
df['Discount_Band'] = pd.cut(
    df['Discount'],
    bins=[-0.01, 0.0, 0.2, 0.4, 1.0],
    labels=['No Discount', 'Low (1-20%)', 'Medium (21-40%)', 'High (41%+)']
)

print("Discount Band distribution:")
print(df['Discount_Band'].value_counts())

Discount Band distribution:
Discount_Band
No Discount        4787
Low (1-20%)        3799
High (41%+)         932
Medium (21-40%)     459
Name: count, dtype: int64


In [14]:
df['Price_Segment'] = df.groupby('Sub-Category')['Unit_Price'].transform(
    lambda x: pd.qcut(x, q=3, labels=['Low', 'Mid', 'Premium'], duplicates='drop')
)

print("Price Segment distribution:")
print(df['Price_Segment'].value_counts())

Price Segment distribution:
Price_Segment
Low        3353
Mid        3328
Premium    3296
Name: count, dtype: int64


In [15]:
negative = df[df['Profit'] < 0]

print(f"Transactions with negative profit : {len(negative)}")
print(f"Percentage of total               : {round(len(negative)/len(df)*100, 2)}%")
print("\nNegative profit count by Category:")
print(negative.groupby('Category')['Profit'].count())

Transactions with negative profit : 1869
Percentage of total               : 18.73%

Negative profit count by Category:
Category
Furniture          713
Office Supplies    885
Technology         271
Name: Profit, dtype: int64


In [16]:
df.to_csv('../data/superstore_cleaned.csv', index=False)
print("Cleaned file saved")

Cleaned file saved


In [18]:
print("        DATA CLEANING SUMMARY")
print(f"Total Rows            : {df.shape[0]}")
print(f"Total Columns         : {df.shape[1]}")
print(f"Categories            : {list(df['Category'].unique())}")
print(f"Sub-Categories        : {df['Sub-Category'].nunique()}")
print(f"Regions               : {list(df['Region'].unique())}")
print(f"Negative Profit Rows  : {len(df[df['Profit'] < 0])}")
print(f"Avg Unit Price        : ${df['Unit_Price'].mean():.2f}")
print(f"Avg Profit Margin     : {df['Profit_Margin'].mean():.2f}%")

        DATA CLEANING SUMMARY
Total Rows            : 9977
Total Columns         : 18
Categories            : ['Furniture', 'Office Supplies', 'Technology']
Sub-Categories        : 17
Regions               : ['South', 'West', 'Central', 'East']
Negative Profit Rows  : 1869
Avg Unit Price        : $60.99
Avg Profit Margin     : 12.01%


In [19]:
print("Final shape:", df.shape)

Final shape: (9977, 18)


In [20]:
print("Unit Price min/max:", df['Unit_Price'].min(), df['Unit_Price'].max())
print("Profit Margin min/max:", df['Profit_Margin'].min(), df['Profit_Margin'].max())

Unit Price min/max: 0.33599999999999997 3773.08
Profit Margin min/max: -275.0 50.0
